# Plan Parameter Operations

In [1]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
USE_LOCAL_SOURCE = False  # <-- TOGGLE THIS

if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)
    if local_path not in sys.path:
        sys.path.insert(0, local_path)
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

# Import ras-commander
from ras_commander import RasCmdr, RasExamples, RasPlan, RasUnsteady, init_ras_project, ras

# Additional imports
import os
import numpy as np
import pandas as pd
from IPython import display
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Verify which version loaded
import ras_commander
print(f"Loaded: {ras_commander.__file__}")

PIP PACKAGE MODE: Loading installed ras-commander


2026-06-11 15:44:30 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


Loaded: <repo>\ras_commander\__init__.py


## Prerequisites

Before running this notebook, ensure you have:

1. **ras-commander installed**: `pip install ras-commander`
2. **Python 3.10+**: Check with `python --version`
3. **HEC-RAS 6.3+**: Required for plan computation
4. **Basic understanding of plan files**: See 103_plan_and_geometry_operations.ipynb

### What You'll Learn

This notebook provides **detailed control over HEC-RAS plan parameters**:

- **Runtime Flags**: Control which calculations HEC-RAS performs
- **Simulation Intervals**: Set output frequency and computation timesteps
- **Descriptions**: Read and write descriptions for plans, geometry, flow, and unsteady files
- **Simulation Dates**: Set start/end times for unsteady simulations
- **DataFrame Access**: Access descriptions directly from project DataFrames

### Related Notebooks

- **103_plan_and_geometry_operations.ipynb** - Plan cloning fundamentals
- **110_single_plan_execution.ipynb** - Execute parameterized plans
- **101_project_initialization.ipynb** - Project structure basics

### Key Concept: Direct Plan File Editing

ras-commander modifies `.p##` files **directly** using text parsing. This provides:
- **Speed**: No GUI automation required
- **Precision**: Exact control over parameters
- **Auditability**: Changes tracked in version control

**Important**: Always verify changes by opening plan in HEC-RAS GUI before execution.

## Parameters

Configure these values to customize the notebook for your project.

In [2]:
# =============================================================================
# PARAMETERS - Edit these to customize the notebook
# =============================================================================
from pathlib import Path

# Project Configuration
PROJECT_NAME = "Muncie"           # Example project to extract
RAS_VERSION = "7.0"               # HEC-RAS version (6.3, 6.5, 6.6, etc.)

## Understanding Plan Files in HEC-RAS

Before we dive into the operations, let's understand what HEC-RAS plan files are and why they're important:

### What is a Plan File?

A HEC-RAS plan file (`.p*`) is a configuration file that defines how a hydraulic simulation will run. It links together:

1. **Geometry**: River channel and floodplain physical characteristics (`.g*` files)
2. **Flow Data**: Inflow conditions, either steady (`.f*`) or unsteady (`.u*`)
3. **Simulation Parameters**: Time steps, computational methods, and output settings

### Key Components of Plan Files

Plan files contain many parameters that control simulation behavior:

- **Simulation Type**: Steady, unsteady, sediment transport, water quality
- **Computation Intervals**: Time steps for calculations
- **Output Intervals**: How frequently results are saved
- **Run Flags**: Which modules to execute (preprocessor, postprocessor, etc.)
- **Simulation Period**: Start and end dates for unsteady simulations
- **Computation Methods**: Numerical schemes and solver settings
- **Resource Allocation**: Number of CPU cores to use

### Why Automate Plan Operations?

Automating plan operations with RAS Commander allows you to:

1. **Batch Processing**: Run multiple scenarios with different parameters
2. **Sensitivity Analysis**: Systematically vary parameters to assess their impact
3. **Calibration**: Adjust parameters to match observed data
4. **Consistency**: Ensure standardized settings across multiple models
5. **Documentation**: Programmatically track simulation configurations

Now, let's download and extract an example project to work with.

In [3]:
# Extract the Bald Eagle Creek example project using static method
bald_eagle_path = RasExamples.extract_project("Balde Eagle Creek", suffix="09")
print(f"Extracted project to: {bald_eagle_path}")

# Verify the path exists
print(f"Bald Eagle Creek project exists: {bald_eagle_path.exists()}")

2026-06-11 15:44:32 - ras_commander.RasExamples - INFO - Successfully extracted project 'Balde Eagle Creek' to <repo>\examples\example_projects\Balde Eagle Creek_09


Extracted project to: <repo>\examples\example_projects\Balde Eagle Creek_09
Bald Eagle Creek project exists: True


## Step 1: Project Initialization

The first step in any RAS Commander workflow is initializing the HEC-RAS project. This connects the Python environment to the HEC-RAS project files.

The `init_ras_project()` function does the following:

1. Locates the main project file (`.prj`)
2. Reads all associated files (plans, geometries, flows)
3. Creates dataframes containing project components
4. Sets up the connection to the HEC-RAS executable

Let's initialize our project:

In [4]:
# Initialize the project (using the default global ras object)
init_ras_project(bald_eagle_path, RAS_VERSION)
print(f"Initialized project: {ras.project_name}")

# Display basic project information
print("\nProject Overview:")
print(f"Project Folder: {ras.project_folder}")
print(f"Project File: {ras.prj_file}")
print(f"Number of Plan Files: {len(ras.plan_df)}")
print(f"Number of Geometry Files: {len(ras.geom_df)}")
print(f"Number of Flow Files: {len(ras.flow_df)}")
print(f"Number of Unsteady Files: {len(ras.unsteady_df)}")

2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 7.0 at C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 5 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 5\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 4 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 4\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.6 at C:\Program Files (x86)\HEC\HEC-RAS\6.6\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.5 at C:\Program Files (x86)\HEC\HEC-RAS\6.5\Ras.exe via filesystem (x86)


...

2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - ras-commander v0.98.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://rascommander.info | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Project initialized: BaldEagle | Folder: <repo>\examples\example_projects\Balde Eagle Creek_09


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/
Repo: https://github.com/gpt-cmdr/ras-commander
═══════════════════════════════════════════════════════════════════════

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow associations
  ras.geom_df        Geometry files and HDF preprocessor paths
  ras.flow_df        Steady flow files
  ras.unsteady_df    Unsteady flow files and configurations
  ras.boundaries_df  Boundary conditions (type, name, location)
  ras.results_df     Lightweight HDF results summaries
  ras.rasmap_df      RASMapper layers, terrain, land cover paths

KEY APIS (static classes — call directly, never instantiate):
  Execution:    RasCmdr.compute_plan() / compute_parallel() / compute_test_mode()
  Plan Files:   RasPlan.clone_plan() / clone_

Initialized project: BaldEagle

Project Overview:
Project Folder: <repo>\examples\example_projects\Balde Eagle Creek_09
Project File: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.prj
Number of Plan Files: 2
Number of Geometry Files: 1
Number of Flow Files: 2
Number of Unsteady Files: 1


Let's also take a look at the plan files in this project:

In [5]:
# Display the plan files
print("Plan Files in Project:")
pd.set_option('display.max_columns', None)
ras.plan_df

Plan Files in Project:


,plan_number,unsteady_number,geometry_number,Plan Title,Program Version,Short Identifier,Simulation Date,Computation Interval,Mapping Interval,Run HTab,Run UNet,Run Sediment,Run PostProcess,Run WQNet,UNET Use Existing IB Tables,Write IC File,Write IC File at Fixed DateTime,IC Time,Write IC File Reoccurance,Write IC File at Sim End,UNET D1 Cores,UNET D2 Cores,PS Cores,DSS File,Friction Slope Method,HDF_Results_Path,Geom File,Geom Path,Flow File,Flow Path,full_path,flow_type
0,01,02,01,Unsteady with Bridges and Dam,5.00,UnsteadyFlow,"18FEB1999,0000,24FEB1999,0500",2MIN,1HOUR,1,1,0,1,0,-1,0,0,",,",,0,0.0,0.0,None,dss,2,None,01,...,02,...,...,Unsteady
1,02,NaN,01,Steady Flow Run,NaN,SteadyRun,"02/18/1999,0000,02/24/1999,0500",2MIN,NaN,1,1,NaN,1,NaN,NaN,0,NaN,,NaN,NaN,NaN,NaN,None,dss,1,None,01,...,02,...,...,Steady


In [6]:
# Get the first plan number for our examples
plan_number = ras.plan_df['plan_number'].iloc[0]
print(f"\nWe'll work with Plan: {plan_number}")


We'll work with Plan: 01


## Step 3: Updating Run Flags

Run flags in HEC-RAS control which components of the simulation are executed. The `RasPlan.update_run_flags()` method allows you to modify these flags programmatically.

### Key Parameters

- `plan_number_or_path` (str or Path): The plan number or full path to the plan file
- `geometry_preprocessor` (bool, optional): Whether to run the geometry preprocessor
- `unsteady_flow_simulation` (bool, optional): Whether to run the unsteady flow simulation
- `run_sediment` (bool, optional): Whether to run sediment transport calculations
- `post_processor` (bool, optional): Whether to run the post-processor
- `floodplain_mapping` (bool, optional): Whether to run floodplain mapping
- `rasect` (RasPrj, optional): The RAS project object

### Common Run Flags

1. **Geometry Preprocessor**: Computes hydraulic tables from geometry data
   - `True`: Recompute tables (useful after geometry changes)
   - `False`: Use existing tables (faster but may be outdated)

2. **Unsteady Flow Simulation**: The main hydraulic calculations
   - `True`: Run unsteady flow calculations
   - `False`: Skip unsteady flow calculations

3. **Sediment Transport**: Simulates erosion and deposition
   - `True`: Calculate sediment transport
   - `False`: Skip sediment transport

4. **Post-Processor**: Calculates additional variables from results
   - `True`: Run post-processing (recommended)
   - `False`: Skip post-processing (faster but fewer outputs)

5. **Floodplain Mapping**: Generates inundation maps
   - `True`: Generate maps (requires terrain data)
   - `False`: Skip mapping (faster)

Let's update the run flags for our plan:

In [7]:
# Update run flags for the plan
print(f"Updating run flags for plan {plan_number}...")
RasPlan.update_run_flags(
    "01",
    geometry_preprocessor=False,     # This may result in a popup if preprocessor files are not present
    unsteady_flow_simulation=False,   # Run the main hydraulic calculations
    run_sediment=False,              # Skip sediment transport calculations
    post_processor=False,             # Run post-processing for additional outputs
    floodplain_mapping=True,        # Skip floodplain mapping
)
print("Run flags updated successfully")

2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Successfully updated run flags in plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01 (flags modified: 5)


Updating run flags for plan 01...
Run flags updated successfully


### Verification: Run Flags Update

**Success Criteria**:
- Plan file modified on disk
- Runtime flags match requested values
- No HEC-RAS warnings when opening plan

**Manual Check**:
```python
# Verify flags were updated
plan_path = RasPlan.get_plan_path(plan_number)

with open(plan_path, 'r') as f:
    content = f.read()

# Check for expected flags
assert 'Run HTab=-1' in content, "HTab flag not set"
assert 'Run RAS=0' in content, "RAS flag not set correctly"
```

**Visual Inspection**:
1. Open project in HEC-RAS GUI
2. Load the modified plan
3. Go to Options menu → Check runtime flags
4. Verify flags match what code set

### Understanding Runtime Flags

| Flag | Meaning | When to Use |
|------|---------|-------------|
| Run HTab=-1 | Compute hydraulic tables | Always for 1D models |
| Run RAS=0 | Don't run steady flow | Unsteady-only plans |
| Run Unsteady=1 | Run unsteady flow | Unsteady plans |
| Write IC File=-1 | Write initial conditions | Before unsteady runs |

See HEC-RAS User's Manual Chapter 5 for complete flag reference.

In [8]:
# The dataframes won't automatically update with changes, so re-init to ensure you are reading the latest version
init_ras_project(bald_eagle_path, RAS_VERSION)

# Display the plan dataframe again to show changes were effective
print("Plan Files in Project:")
ras.plan_df

2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 7.0 at C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 5 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 5\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 4 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 4\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.6 at C:\Program Files (x86)\HEC\HEC-RAS\6.6\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.5 at C:\Program Files (x86)\HEC\HEC-RAS\6.5\Ras.exe via filesystem (x86)


...

2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Project initialized: BaldEagle | Folder: <repo>\examples\example_projects\Balde Eagle Creek_09


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/
Repo: https://github.com/gpt-cmdr/ras-commander
═══════════════════════════════════════════════════════════════════════

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow associations
  ras.geom_df        Geometry files and HDF preprocessor paths
  ras.flow_df        Steady flow files
  ras.unsteady_df    Unsteady flow files and configurations
  ras.boundaries_df  Boundary conditions (type, name, location)
  ras.results_df     Lightweight HDF results summaries
  ras.rasmap_df      RASMapper layers, terrain, land cover paths

KEY APIS (static classes — call directly, never instantiate):
  Execution:    RasCmdr.compute_plan() / compute_parallel() / compute_test_mode()
  Plan Files:   RasPlan.clone_plan() / clone_

Plan Files in Project:


,plan_number,unsteady_number,geometry_number,Plan Title,Program Version,Short Identifier,Simulation Date,Computation Interval,Mapping Interval,Run HTab,Run UNet,Run Sediment,Run PostProcess,Run WQNet,UNET Use Existing IB Tables,Write IC File,Write IC File at Fixed DateTime,IC Time,Write IC File Reoccurance,Write IC File at Sim End,UNET D1 Cores,UNET D2 Cores,PS Cores,DSS File,Friction Slope Method,HDF_Results_Path,Geom File,Geom Path,Flow File,Flow Path,full_path,flow_type
0,01,02,01,Unsteady with Bridges and Dam,5.00,UnsteadyFlow,"18FEB1999,0000,24FEB1999,0500",2MIN,1HOUR,0,0,0,0,0,-1,0,0,",,",,0,0.0,0.0,None,dss,2,None,01,...,02,...,...,Unsteady
1,02,NaN,01,Steady Flow Run,NaN,SteadyRun,"02/18/1999,0000,02/24/1999,0500",2MIN,NaN,1,1,NaN,1,NaN,NaN,0,NaN,,NaN,NaN,NaN,NaN,None,dss,1,None,01,...,02,...,...,Steady


In [9]:
plan_path = RasPlan.get_plan_path("01")


In [10]:
print(plan_path)

<repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01


In [11]:
# Print the Plan file's contents to confirm the change

# Print the plan file contents to verify the run flag changes
with open(plan_path, 'r') as f:
    print(f.read())



Plan Title=Unsteady with Bridges and Dam
Program Version=5.00
Short Identifier=UnsteadyFlow                                                    
Simulation Date=18FEB1999,0000,24FEB1999,0500
Geom File=g01
Flow File=u02
Subcritical Flow
K Sum by GR= 0 
Std Step Tol= 0.01 
Critical Tol= 0.01 
Num of Std Step Trials= 20 
Max Error Tol= 0.3 
Flow Tol Ratio= 0.001 
Split Flow NTrial= 30 
Split Flow Tol= 0.02 
Split Flow Ratio= 0.02 
Log Output Level= 0 
Friction Slope Method= 2 
Unsteady Friction Slope Method= 2 
Unsteady Bridges Friction Slope Method= 1 
Parabolic Critical Depth
Global Vel Dist= 0 , 0 , 0 
Global Log Level= 0 
CheckData=True
Encroach Param=-1 ,0,0, 0 
Computation Interval=2MIN
Output Interval=1HOUR
Instantaneous Interval=2HOUR
Mapping Interval=1HOUR
Run HTab= 0
Run UNet= 0
Run Sediment= 0
Run PostProcess= 0
Run WQNet= 0 
Run RASMapper= -1
UNET Theta= 1 
UNET Theta Warmup= 1 
UNET ZTol= 0.01 
UNET ZSATol= 0.1 
UNET QTol=
UNET MxIter= 20 
UNET Max Iter WO Improvement= 0 
UNET

## Step 4: Updating Plan Intervals

Time intervals in HEC-RAS control the temporal resolution of simulations and outputs. The `RasPlan.update_plan_intervals()` method allows you to modify these intervals.

### Key Parameters

- `plan_number_or_path` (str or Path): The plan number or full path to the plan file
- `computation_interval` (str, optional): Time step for calculations
- `output_interval` (str, optional): Time step for saving detailed results
- `instantaneous_interval` (str, optional): Time step for peak value calculations
- `mapping_interval` (str, optional): Time step for map outputs
- `rasect` (RasPrj, optional): The RAS project object

### Valid Interval Values

Time intervals must be specified in HEC-RAS format:
- Seconds: `1SEC`, `2SEC`, `3SEC`, `4SEC`, `5SEC`, `6SEC`, `10SEC`, `15SEC`, `20SEC`, `30SEC`
- Minutes: `1MIN`, `2MIN`, `3MIN`, `4MIN`, `5MIN`, `6MIN`, `10MIN`, `15MIN`, `20MIN`, `30MIN`
- Hours: `1HOUR`, `2HOUR`, `3HOUR`, `4HOUR`, `6HOUR`, `8HOUR`, `12HOUR`
- Days: `1DAY`

### Interval Types

1. **Computation Interval**: Time step used for hydraulic calculations
   - Smaller intervals: More accurate but slower
   - Larger intervals: Faster but may introduce numerical errors
   - Rule of thumb: Should be small enough to capture flow changes

2. **Output Interval**: How frequently detailed results are saved
   - Smaller intervals: More detailed results but larger files
   - Larger intervals: Smaller files but less temporal resolution
   - Usually larger than computation interval

3. **Instantaneous Interval**: Time step for peak value calculations
   - Affects when max/min values are checked
   - Usually equal to output interval

4. **Mapping Interval**: How frequently map data is saved
   - Affects animation smoothness and file size
   - Usually larger than output interval

Let's update the intervals for our plan:

In [12]:
# Update plan intervals
print(f"Updating intervals for plan {plan_number}...")
RasPlan.update_plan_intervals(
    plan_number,
    computation_interval="5SEC",    # 5-second time step for calculations
    output_interval="1MIN",         # Save detailed results every minute
    instantaneous_interval="5MIN",  # Check for max/min values every 5 minutes
    mapping_interval="15MIN",       # Save map data every 15 minutes

)
print("Plan intervals updated successfully")

2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Successfully updated intervals in plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01


Updating intervals for plan 01...
Plan intervals updated successfully


## Step 5: Managing Descriptions Across All File Types

HEC-RAS files use a standard `BEGIN DESCRIPTION / END DESCRIPTION` block for documentation. ras-commander provides read/write methods for **all four file types**: plans, geometries, steady flow, and unsteady flow.

### Available Methods

| File Type | Read | Write |
|-----------|------|-------|
| **Plan** (`.p##`) | `RasPlan.read_plan_description()` | `RasPlan.update_plan_description()` |
| **Geometry** (`.g##`) | `RasPlan.read_geom_description()` | `RasPlan.update_geom_description()` |
| **Steady Flow** (`.f##`) | `RasPlan.read_flow_description()` | `RasPlan.update_flow_description()` |
| **Unsteady Flow** (`.u##`) | `RasUnsteady.read_unsteady_description()` | `RasUnsteady.update_unsteady_description()` |

All methods accept either a file number (e.g., `"01"`) or a full file path.

### DataFrame Columns

After project initialization, descriptions are available directly in DataFrames:
- `ras.plan_df['description']` - Plan descriptions
- `ras.geom_df['description']` and `ras.geom_df['geom_title']` - Geometry info
- `ras.flow_df['description']` - Steady flow descriptions
- `ras.unsteady_df['description']` - Unsteady flow descriptions

Let's demonstrate reading and writing descriptions for each file type.

In [13]:
# --- Plan Description ---
print("=" * 60)
print("PLAN DESCRIPTION")
print("=" * 60)

# Read the current plan description
current_description = RasPlan.read_plan_description(plan_number)
print(f"Current plan description:\n'{current_description}'\n")

# Create a new description with detailed information
new_description = f"""Modified Plan for RAS Commander Testing
Date: {datetime.now().strftime('%Y-%m-%d')}
Purpose: Demonstrating RAS Commander plan operations
Settings:
- Computation Interval: 5SEC
- Output Interval: 1MIN
- Mapping Interval: 15MIN
Notes: This plan was automatically modified using ras-commander."""

# Update the plan description
print("Updating plan description...")
result = RasPlan.update_plan_description(plan_number, new_description)
print(f"Update result: {result}")

# Verify the updated description reads back correctly
updated_description = RasPlan.read_plan_description(plan_number)
print(f"\nUpdated plan description:\n{updated_description}")

2026-06-11 15:44:32 - ras_commander.RasPlan - WARNING - No description found in plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01


PLAN DESCRIPTION


2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Read description from plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01


Current plan description:
''

Updating plan description...
Update result: True



Updated plan description:
Modified Plan for RAS Commander Testing
Date: 2026-06-11
Purpose: Demonstrating RAS Commander plan operations
Settings:
- Computation Interval: 5SEC
- Output Interval: 1MIN
- Mapping Interval: 15MIN
Notes: This plan was automatically modified using ras-commander.


In [14]:
# --- Geometry Description ---
print("=" * 60)
print("GEOMETRY DESCRIPTION")
print("=" * 60)

# Read current geometry description (geometry number "01")
geom_desc = RasPlan.read_geom_description("01")
print(f"Current geometry description:\n'{geom_desc}'\n")

# Update geometry description
geom_description = f"""Bald Eagle Creek Geometry
Modified: {datetime.now().strftime('%Y-%m-%d')}
Contains 1D cross sections and inline structures (bridges, dam)."""

result = RasPlan.update_geom_description("01", geom_description)
print(f"Update result: {result}")

# Verify
updated_geom_desc = RasPlan.read_geom_description("01")
print(f"\nUpdated geometry description:\n{updated_geom_desc}")

2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Found geometry path: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.g01


2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Found geometry path: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.g01


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Updated description in <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.g01


2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Found geometry path: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.g01


GEOMETRY DESCRIPTION
Current geometry description:
'Foster Joseph Sayers Dam and Reservoir'

Update result: True

Updated geometry description:
Bald Eagle Creek Geometry
Modified: 2026-06-11
Contains 1D cross sections and inline structures (bridges, dam).


In [15]:
# --- Steady Flow Description ---
print("=" * 60)
print("STEADY FLOW DESCRIPTION")
print("=" * 60)

# Check which flow files exist
print("Flow files in project:")
print(ras.flow_df[['flow_number', 'full_path']].to_string())
print()

# Read current flow description (use first flow file)
flow_num = ras.flow_df['flow_number'].iloc[0]
flow_desc = RasPlan.read_flow_description(flow_num)
print(f"Current flow description for f{flow_num}:\n'{flow_desc}'\n")

# Update flow description
flow_description = f"""Steady flow data for Bald Eagle Creek
Modified: {datetime.now().strftime('%Y-%m-%d')}
Contains baseline flow profiles for calibration."""

result = RasPlan.update_flow_description(flow_num, flow_description)
print(f"Update result: {result}")

# Verify
updated_flow_desc = RasPlan.read_flow_description(flow_num)
print(f"\nUpdated flow description:\n{updated_flow_desc}")

2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Updated description in <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.f02


STEADY FLOW DESCRIPTION
Flow files in project:
  flow_number                                                                                                     full_path
0          02  <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.f02
1          01  <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.f01

Current flow description for f02:
''

Update result: True

Updated flow description:
Steady flow data for Bald Eagle Creek
Modified: 2026-06-11
Contains baseline flow profiles for calibration.


In [16]:
# --- Unsteady Flow Description ---
print("=" * 60)
print("UNSTEADY FLOW DESCRIPTION")
print("=" * 60)

# Check which unsteady files exist
print("Unsteady files in project:")
print(ras.unsteady_df[['unsteady_number', 'full_path']].to_string())
print()

# Read current unsteady description
unsteady_num = ras.unsteady_df['unsteady_number'].iloc[0]
unsteady_desc = RasUnsteady.read_unsteady_description(unsteady_num)
print(f"Current unsteady description for u{unsteady_num}:\n'{unsteady_desc}'\n")

# Update unsteady description
unsteady_description = f"""Unsteady flow boundary conditions for Bald Eagle Creek
Modified: {datetime.now().strftime('%Y-%m-%d')}
Event: February 1999 flood event
Upstream: Flow hydrograph, Downstream: Normal depth."""

result = RasUnsteady.update_unsteady_description(unsteady_num, unsteady_description)
print(f"Update result: {result}")

# Verify
updated_unsteady_desc = RasUnsteady.read_unsteady_description(unsteady_num)
print(f"\nUpdated unsteady description:\n{updated_unsteady_desc}")

2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Updated description in <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.u02


UNSTEADY FLOW DESCRIPTION
Unsteady files in project:
  unsteady_number                                                                                                     full_path
0              02  <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.u02

Current unsteady description for u02:
''

Update result: True

Updated unsteady description:
Unsteady flow boundary conditions for Bald Eagle Creek
Modified: 2026-06-11
Event: February 1999 flood event
Upstream: Flow hydrograph, Downstream: Normal depth.


In [17]:
# --- Verify descriptions in DataFrames ---
# Re-initialize to pick up all description changes
init_ras_project(bald_eagle_path, RAS_VERSION)

print("=" * 60)
print("DESCRIPTIONS IN DATAFRAMES (after re-initialization)")
print("=" * 60)

# Plan descriptions
print("\nplan_df 'description' column:")
for _, row in ras.plan_df.iterrows():
    desc = row.get('description', 'N/A')
    print(f"  Plan {row['plan_number']}: {str(desc)[:80]}...")

# Geometry descriptions and titles
print("\ngeom_df 'geom_title' and 'description' columns:")
for _, row in ras.geom_df.iterrows():
    title = row.get('geom_title', 'N/A')
    desc = row.get('description', 'N/A')
    print(f"  Geom {row['geom_number']}: title='{title}', description='{str(desc)[:60]}...'")

# Flow descriptions
print("\nflow_df 'description' column:")
for _, row in ras.flow_df.iterrows():
    desc = row.get('description', 'N/A')
    print(f"  Flow {row['flow_number']}: {str(desc)[:80]}...")

# Unsteady descriptions
print("\nunsteady_df 'description' column:")
for _, row in ras.unsteady_df.iterrows():
    desc = row.get('description', 'N/A')
    print(f"  Unsteady {row['unsteady_number']}: {str(desc)[:80]}...")

2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 7.0 at C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 5 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 5\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.7 Beta 4 at C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 4\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.6 at C:\Program Files (x86)\HEC\HEC-RAS\6.6\Ras.exe via filesystem (x86)


2026-06-11 15:44:32 - ras_commander.RasUtils - INFO - Discovered HEC-RAS 6.5 at C:\Program Files (x86)\HEC\HEC-RAS\6.5\Ras.exe via filesystem (x86)


...

2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - ras-commander v0.98.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://rascommander.info | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Project initialized: BaldEagle | Folder: <repo>\examples\example_projects\Balde Eagle Creek_09


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - Using HEC-RAS executable: C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe


2026-06-11 15:44:32 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/
Repo: https://github.com/gpt-cmdr/ras-commander
═══════════════════════════════════════════════════════════════════════

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow associations
  ras.geom_df        Geometry files and HDF preprocessor paths
  ras.flow_df        Steady flow files
  ras.unsteady_df    Unsteady flow files and configurations
  ras.boundaries_df  Boundary conditions (type, name, location)
  ras.results_df     Lightweight HDF results summaries
  ras.rasmap_df      RASMapper layers, terrain, land cover paths

KEY APIS (static classes — call directly, never instantiate):
  Execution:    RasCmdr.compute_plan() / compute_parallel() / compute_test_mode()
  Plan Files:   RasPlan.clone_plan() / clone_

DESCRIPTIONS IN DATAFRAMES (after re-initialization)

plan_df 'description' column:
  Plan 01: Modified Plan for RAS Commander Testing
Date: 2026-06-11
Purpose: Demonstrating ...
  Plan 02: nan...

geom_df 'geom_title' and 'description' columns:
  Geom 01: title='Existing Conditions - GIS Data', description='Bald Eagle Creek Geometry
Modified: 2026-06-11
Contains 1D c...'

flow_df 'description' column:
  Flow 02: Steady flow data for Bald Eagle Creek
Modified: 2026-06-11
Contains baseline flo...
  Flow 01: ...

unsteady_df 'description' column:
  Unsteady 02: Unsteady flow boundary conditions for Bald Eagle Creek
Modified: 2026-06-11
Even...


## Step 6: Updating Simulation Dates

For unsteady flow simulations, the simulation period defines the time window for the analysis. The `RasPlan.update_simulation_date()` method allows you to modify this period.

### Key Parameters

- `plan_number_or_path` (str or Path): The plan number or full path to the plan file
- `start_date` (datetime): The start date and time for the simulation
- `end_date` (datetime): The end date and time for the simulation
- `rasect` (RasPrj, optional): The RAS project object

### Considerations for Simulation Dates

1. **Hydrograph Coverage**: The simulation period should fully encompass your hydrographs
2. **Warm-Up Period**: Include time before the main event for model stabilization
3. **Cool-Down Period**: Include time after the main event for complete drainage
4. **Computational Efficiency**: Avoid unnecessarily long periods to reduce runtime
5. **Consistency**: Ensure dates match available boundary condition data

Let's update the simulation dates for our plan:

In [18]:
# Get the current simulation date
current_sim_date = RasPlan.get_plan_value(plan_number, "Simulation Date")
print(f"Current simulation date: {current_sim_date}")

# Parse the current simulation date string
current_dates = current_sim_date.split(",")
current_start = datetime.strptime(f"{current_dates[0]},{current_dates[1]}", "%d%b%Y,%H%M")
current_end = datetime.strptime(f"{current_dates[2]},{current_dates[3]}", "%d%b%Y,%H%M")

# Define new simulation period - adjust by 1 hour from current dates
start_date = current_start + timedelta(hours=1)  # Current start + 1 hour
end_date = current_end - timedelta(hours=1)      # Current end - 1 hour

# Update the simulation date
print(f"\nUpdating simulation period to: {start_date.strftime('%d%b%Y,%H%M')} - {end_date.strftime('%d%b%Y,%H%M')}")
RasPlan.update_simulation_date(plan_number, start_date, end_date)
print("Simulation dates updated successfully")

# Verify the updated simulation date
updated_sim_date = RasPlan.get_plan_value(plan_number, "Simulation Date")
print(f"\nUpdated simulation date: {updated_sim_date}")

Current simulation date: 18FEB1999,0000,24FEB1999,0500

Updating simulation period to: 18Feb1999,0100 - 24Feb1999,0400


2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Updated simulation date in plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01


Simulation dates updated successfully

Updated simulation date: 18FEB1999,0100,24FEB1999,0400


## Step 7: Verifying Updated Plan Values

After making multiple changes to a plan, it's a good practice to verify that all updates were applied correctly. Let's check the updated values:

In [19]:
# Print the Plan file's contents to confirm the change

# Print the plan file contents to verify the run flag changes
with open(plan_path, 'r') as f:
    print(f.read())



Plan Title=Unsteady with Bridges and Dam
Program Version=5.00
Short Identifier=UnsteadyFlow                                                    
Simulation Date=18FEB1999,0100,24FEB1999,0400
Geom File=g01
Flow File=u02
Subcritical Flow
K Sum by GR= 0 
Std Step Tol= 0.01 
Critical Tol= 0.01 
Num of Std Step Trials= 20 
Max Error Tol= 0.3 
Flow Tol Ratio= 0.001 
Split Flow NTrial= 30 
Split Flow Tol= 0.02 
Split Flow Ratio= 0.02 
Log Output Level= 0 
Friction Slope Method= 2 
Unsteady Friction Slope Method= 2 
Unsteady Bridges Friction Slope Method= 1 
Parabolic Critical Depth
Global Vel Dist= 0 , 0 , 0 
Global Log Level= 0 
CheckData=True
Encroach Param=-1 ,0,0, 0 
BEGIN DESCRIPTION:
Modified Plan for RAS Commander Testing
Date: 2026-06-11
Purpose: Demonstrating RAS Commander plan operations
Settings:
- Computation Interval: 5SEC
- Output Interval: 1MIN
- Mapping Interval: 15MIN
Notes: This plan was automatically modified using ras-commander.
END DESCRIPTION:
Computation Interval=5SEC
Ou

## Step 8: Computing the Plan (Optional)

After making changes to a plan, you might want to run the simulation to see the effects. The `RasCmdr.compute_plan()` method executes a HEC-RAS simulation with the specified plan.

### Key Parameters

- `plan_number` (str): The plan number to execute
- `dest_folder` (str, Path, optional): Destination folder for computation
- `rasect` (RasPrj, optional): The RAS project object
- `clear_geompre` (bool, optional): Whether to clear geometry preprocessor files
- `num_cores` (int, optional): Number of processor cores to use
- `overwrite_dest` (bool, optional): Whether to overwrite the destination folder

If you want to run the simulation, you can uncomment the code below:

In [20]:
RasPlan.update_run_flags(plan_number, geometry_preprocessor=True)
RasCmdr.compute_plan(plan_number)

2026-06-11 15:44:32 - ras_commander.RasPlan - INFO - Successfully updated run flags in plan file: <repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01 (flags modified: 1)


2026-06-11 15:44:32 - ras_commander.RasCmdr - INFO - Using ras_object with project folder: <repo>\examples\example_projects\Balde Eagle Creek_09


2026-06-11 15:44:32 - ras_commander.RasCmdr - INFO - Running HEC-RAS from the Command Line:


2026-06-11 15:44:32 - ras_commander.RasCmdr - INFO - Running command: "C:\Program Files (x86)\HEC\HEC-RAS\7.0\Ras.exe" -c "<repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.prj" "<repo>\examples\example_projects\Balde Eagle Creek_09\BaldEagle.p01"


2026-06-11 15:44:32 - ras_commander.RasDialogWatchdog - INFO - DialogWatchdog started — polling every 1.5s for RAS dialog windows


2026-06-11 15:45:49 - ras_commander.RasCmdr - INFO - HEC-RAS execution completed for plan: 01


2026-06-11 15:45:49 - ras_commander.RasCmdr - INFO - Total run time for plan 01: 76.28 seconds


2026-06-11 15:45:49 - ras_commander.RasDialogWatchdog - INFO - DialogWatchdog stopped — no dialogs encountered


2026-06-11 15:45:49 - ras_commander.hdf.HdfResultsPlan - WARNING - Group '/Plan Data/Plan Information' not found.


ComputeResult(SUCCESS, results_df_row=available)

In [21]:
# Uncomment to run the simulation with the updated plan

# # Define a destination folder for the computation
# dest_folder = script_dir / "compute_results"
# print(f"Computing plan {plan_number}...")
# print(f"Results will be saved to: {dest_folder}")

# # Execute the plan
# success = RasCmdr.compute_plan(
#     plan_number,
#     dest_folder=dest_folder,
#     clear_geompre=True,    # Clear preprocessor files to ensure clean results
#     num_cores=2,           # Use 2 processor cores
#     overwrite_dest=True,   # Overwrite existing destination folder
#     rasect=ras
# )

# if success:
#     print(f"Plan {plan_number} computed successfully")
#     # Check for results file
#     results_path = RasPlan.get_results_path(plan_number)
#     if results_path:
#         print(f"Results saved to: {results_path}")
# else:
#     print(f"Failed to compute plan {plan_number}")

## Viewing Execution Summary with results_df

Check how parameter changes affected execution performance.

In [22]:
# Display execution summary only for the plan run in this notebook (filtered and transposed)
print("Execution Summary (for current plan):")
display.display(ras.results_df[ras.results_df['plan_number'] == plan_number].T)

Execution Summary (for current plan):


,1
plan_number,01
plan_title,Unsteady with Bridges and Dam
flow_type,Unsteady
hdf_path,...
hdf_exists,True
completed,True
has_errors,False
has_warnings,False
error_count,0
warning_count,0


## HEC-RAS Plan Parameters Reference

### Parameter Documentation

- **HEC-RAS User's Manual - Chapter 5**: Simulation window options
  https://www.hec.usace.army.mil/software/hec-ras/documentation.aspx
- **Technical Reference - Section 2**: Computation methods and intervals

### Common Parameters

**Computation Intervals**:
- `Computation Interval`: Timestep for hydraulic calculations (seconds)
- `Output Interval`: Frequency of results saved to HDF (seconds)
- `Mapping Interval`: Frequency for RAS Mapper output (seconds)

**Simulation Timing**:
- `Simulation Date`: Start date/time (format: DDMMMYYYY,HHMM)
- `End Date`: End of simulation period

### LLM Forward: Parameter Change Tracking

Create audit trail for parameter modifications:

```python
def audit_parameter_changes(plan_number, changes_dict, output_file):
    # Document all parameter changes
    audit_record = {
        'plan_number': plan_number,
        'timestamp': datetime.now().isoformat(),
        'changes': []
    }

    for param_name, (old_val, new_val) in changes_dict.items():
        audit_record['changes'].append({
            'parameter': param_name,
            'original_value': str(old_val),
            'new_value': str(new_val),
            'change_type': type(new_val).__name__
        })

    # Export audit record
    import json
    with open(output_file, 'w') as f:
        json.dump(audit_record, f, indent=2)

    print(f"Parameter changes documented: {output_file}")

# Usage
changes = {
    'Computation Interval': (10, 5),
    'Output Interval': (60, 30),
    'Mapping Interval': (3600, 1800)
}

audit_parameter_changes(
    plan_number,
    changes,
    project_folder / 'parameter_audit.json'
)
```

This provides:
- **Change history**: Know what was modified when
- **Reproducibility**: Revert changes if needed
- **Peer review**: Non-programmers can verify modifications

## Summary

In this notebook, we covered essential operations for manipulating HEC-RAS plan files programmatically:

1. **Project Initialization**: `init_ras_project()` populates DataFrames with project metadata
2. **Run Flags**: `RasPlan.update_run_flags()` controls which HEC-RAS components execute
3. **Plan Intervals**: `RasPlan.update_plan_intervals()` sets computation and output timesteps
4. **Descriptions for All File Types**:
   - Plans: `RasPlan.read_plan_description()` / `update_plan_description()`
   - Geometry: `RasPlan.read_geom_description()` / `update_geom_description()`
   - Steady Flow: `RasPlan.read_flow_description()` / `update_flow_description()`
   - Unsteady Flow: `RasUnsteady.read_unsteady_description()` / `update_unsteady_description()`
5. **Simulation Dates**: `RasPlan.update_simulation_date()` for unsteady analysis periods
6. **DataFrame Access**: Descriptions available via `ras.plan_df['description']`, `ras.geom_df['description']`, `ras.geom_df['geom_title']`, `ras.flow_df['description']`, `ras.unsteady_df['description']`

### Key Classes

| Class | Methods |
|-------|---------|
| `RasPlan` | `get_plan_value()`, `update_run_flags()`, `update_plan_intervals()`, `read_plan_description()`, `update_plan_description()`, `read_geom_description()`, `update_geom_description()`, `read_flow_description()`, `update_flow_description()`, `update_simulation_date()` |
| `RasUnsteady` | `read_unsteady_description()`, `update_unsteady_description()` |
| `RasCmdr` | `compute_plan()` |

### Related Notebooks

- **103_plan_and_geometry_operations.ipynb** - Plan cloning fundamentals
- **110_single_plan_execution.ipynb** - Execute parameterized plans
- **101_project_initialization.ipynb** - Project structure basics